In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS aulas_ao_vivo.live_20260302;

GRANT ALL PRIVILEGES ON CATALOG aulas_ao_vivo TO `marcocaja.dataengineer@gmail.com`; -- só se já não for dono full do catálogo

In [0]:
# Live 1 (P) — Limpeza Mínima v1
# Databricks + PySpark | Dataset sintético (spark.createDataFrame / spark.range)

from pyspark.sql import functions as F
from pyspark.sql import types as T

try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    
    # em caso de estar fora do Databricks, por qualquer motivo que seja
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("Live1_Limpeza_Minima") \
        .config("spark.sql.shuffle.partitions", "8") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("WARN")

# Parâmetros didáticos (ajuste para ver o antes/depois em tamanhos diferentes)
N_LINHAS = 50000
SEED = 66
START_TS = "2026-03-01 00:00:00"

def d(df, n=20):
    """Helper de visualização: usa display() no Databricks; senão, df.show()."""
    try:
        display(df)
    except NameError:
        df.show(n, truncate=False)

def standardize_column_names(df):
    """
    Padroniza nomes de colunas para snake_case.
    Ex.: "Valor Total (R$)" -> "valor_total_r" (depois renomeamos para o nome canônico)
    """
    import re
    cols = df.columns
    new_cols = []
    for c in cols:
        c2 = c.strip().lower()
        c2 = re.sub(r"[^a-z0-9]+", "_", c2)
        c2 = re.sub(r"_+", "_", c2).strip("_")
        new_cols.append(c2)
    out = df
    for old, new in zip(cols, new_cols):
        if old != new:
            out = out.withColumnRenamed(old, new)
    return out


In [0]:
from random import randint
def generate_vendas_bronze(n_linhas: int, seed: int = 42, start_ts: str = "2026-03-01 00:00:00"):
    """
    Gera um DataFrame propositalmente "sujo" (bronze) para demonstrar limpeza mínima.
    Controle de volume: n_linhas.

    Colunas (intencionalmente inconsistentes):
      - "Id Pedido" (string com espaços)
      - "Data Pedido" (string em formatos mistos)
      - "UF " (string com caixa/espaços inconsistentes)
      - "Produto" (string com caixa/espaços inconsistentes)
      - "Qtd" (string numérica com zeros/espaços)
      - "Valor Total (R$)" (string com separador decimal misto)
      - "Status" (string com variações)
    """
    if n_linhas < 1:
        return spark.createDataFrame([], schema=T.StructType([]))

    ufs = [
        "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA",
        "PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"
    ]
    produtos = ["Camiseta", "Tenis", "Caderno", "Mochila", "Fone", "Mouse", "Garrafa", "Jaqueta"]

    base = spark.range(n_linhas).withColumn("rnd", F.rand(seed))

    # id_pedido com alguns espaços extras
    id_raw = F.concat(F.lit("PED"), F.lpad(F.col("id").cast("string"), 8, "0"))
    id_sujo = F.when(F.col("id") % 5 == 0, F.concat(F.lit(" "), id_raw, F.lit(" "))).otherwise(id_raw)

    # timestamp base (incrementos de 1 hora) e representações em string com formatos mistos
    start_unix = F.unix_timestamp(F.lit(start_ts), "yyyy-MM-dd HH:mm:ss")
    ts = F.to_timestamp(F.from_unixtime(start_unix + (F.col("id") * F.lit(3600))))
    fmt1 = F.date_format(ts, "yyyy-MM-dd HH:mm:ss")
    fmt2 = F.date_format(ts, "dd/MM/yyyy HH:mm")
    fmt3 = F.date_format(ts, "dd/MM/yyyy HH:mm:ss")
    data_suja = (
        F.when(F.col("id") % 3 == 0, fmt1)
         .when(F.col("id") % 3 == 1, fmt2)
         .otherwise(fmt3)
    )

    # Índices do element_at precisam ser INT (spark.range cria id como BIGINT)
    idx_uf = ((F.col("id") % F.lit(len(ufs))) + F.lit(1)).cast("int")
    idx_prod = ((F.col("id") % F.lit(len(produtos))) + F.lit(1)).cast("int")

    # UF com caixa e espaços inconsistentes
    uf_raw = F.element_at(F.array(*[F.lit(x) for x in ufs]), idx_uf)
    uf_sujo = (
        F.when(F.col("id") % 4 == 0, F.lower(uf_raw))
         .when(F.col("id") % 4 == 1, F.concat(F.lit(" "), uf_raw, F.lit(" ")))
         .otherwise(uf_raw)
    )

    # Produto com variações de caixa/espaços
    prod_raw = F.element_at(F.array(*[F.lit(x) for x in produtos]), idx_prod)
    prod_sujo = (
        F.when(F.col("id") % 4 == 0, F.upper(prod_raw))
         .when(F.col("id") % 4 == 1, F.concat(F.lit(" "), prod_raw, F.lit(" ")))
         .when(F.col("id") % 7 == 0, F.lit(" "))
         .otherwise(prod_raw)
    )

    # Quantidade como string (com zero à esquerda e espaços)
    qty_num = (F.col("id") % F.lit(5)) + F.lit(1)
    qty_str = (
        F.when(F.col("id") % 4 == 0, F.lpad(qty_num.cast("string"), 2, "0"))
         .when(F.col("id") % 9 == 0, F.lit(" "))
         .otherwise(qty_num.cast("string"))
    )

    # Valor total como string (alguns com vírgula como separador decimal)
    preco_base = ((F.col("id") % F.lit(20)) + F.lit(1)) * F.lit(10.5)
    valor_num = F.round(preco_base * qty_num.cast("double"), 2)
    valor_dot = F.format_string("%.2f", valor_num)  # ex: 1234.56
    valor_comma = F.regexp_replace(valor_dot, "\\.", ",")  # ex: 1234,56
    valor_sujo = (
        F.when(F.col("id") % 3 == 0, valor_dot)
         .when(F.col("id") % 3 == 1, valor_comma)
         .otherwise(F.concat(F.lit(" "), valor_comma, F.lit(" ")))
    )

    # Status com variações
    status_sujo = (
        F.when(F.col("id") % 3 == 0, F.lit("Pago"))
         .when(F.col("id") % 3 == 1, F.lit("CANCELADO"))
         .otherwise(F.lit(" Pendente "))
    )

    df = base.select(
        id_sujo.alias("Id Pedido"),
        data_suja.alias("Data Pedido"),
        uf_sujo.alias("UF "),
        prod_sujo.alias("Produto"),
        qty_str.alias("Qtd"),
        valor_sujo.alias("Valor Total (R$)"),
        status_sujo.alias("Status")
    )

    return df


In [0]:
# 1) Criar dataset pequeno (ANTES)
vendas_bronze = generate_vendas_bronze(N_LINHAS, seed=SEED, start_ts=START_TS)

print("--- Schema (BRONZE / antes) ---")
vendas_bronze.printSchema()

print("--- Amostra (BRONZE / antes) ---")
d(vendas_bronze.limit(20))

print("Colunas originais:", vendas_bronze.columns)


In [0]:
# 2) Passo: padronizar nomes de colunas (snake_case)
"""
PASSO: Padronização de nomes de colunas
O QUE OBSERVAR: colunas com espaços, parênteses e caixa misturada viram snake_case.
VALIDAR: se ficou legível e consistente (sem espaços / sem caracteres especiais).
SINAL DE ERRO: colunas duplicadas após normalização.
"""

vendas_1 = standardize_column_names(vendas_bronze)
print("Colunas padronizadas:", vendas_1.columns)
vendas_1.printSchema()
d(vendas_1.limit(10))


In [0]:
# 3) Passo: renomear para nomes canônicos (contrato do mês)
"""
PASSO: Renomear colunas para nomes canônicos
VALIDAR: se as colunas finais existem e fazem sentido.
"""

rename_map = {
    "qtd": "quantidade",
    "valor_total_r": "valor_total"
}

vendas_2 = vendas_1
for old, new in rename_map.items():
    if old in vendas_2.columns:
        vendas_2 = vendas_2.withColumnRenamed(old, new)

print("Colunas canônicas:", vendas_2.columns)
d(vendas_2.limit(10))


In [0]:
# 4) Separar datas em três DataFrames conforme formato
# 4.1) trim em todas as colunas string
v = vendas_2
for c, t in v.dtypes:
    if t == "string":
        v = v.withColumn(c, F.trim(F.col(c)))

# 4.2) Normalização de colunas-chave
v = (
    v
    .withColumn("id_pedido", F.upper(F.col("id_pedido")))
    .withColumn("uf", F.upper(F.col("uf")))
    .withColumn("produto", F.lower(F.col("produto")))
    .withColumn("status", F.lower(F.col("status")))
)

# 4.3) Categorias: status -> {pago, cancelado, pendente}
v = v.withColumn(
    "status",
    F.when(F.col("status").rlike("cancel"), F.lit("cancelado"))
     .when(F.col("status").rlike("pend"), F.lit("pendente"))
     .otherwise(F.lit("pago"))
)

# 4.4) Split datas por formato
v_iso = v.filter(F.col("data_pedido").rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"))
# dd/MM/yyyy HH:mm:ss
v_br_full = v.filter(F.col("data_pedido").rlike(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2}$"))
# dd/MM/yyyy HH:mm
v_br_short = v.filter(F.col("data_pedido").rlike(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$"))

# 4.5) Cast para timestamp conforme formato
v_iso = v_iso.withColumn("data_pedido", F.to_timestamp(F.col("data_pedido"), "yyyy-MM-dd HH:mm:ss"))
v_br_full = v_br_full.withColumn("data_pedido", F.to_timestamp(F.col("data_pedido"), "dd/MM/yyyy HH:mm:ss"))
v_br_short = v_br_short.withColumn(
    "data_pedido", F.to_timestamp(F.concat(F.col("data_pedido"), F.lit(":00")), "dd/MM/yyyy HH:mm:ss")
)

# 4.6) Union dos DataFrames
v_all = v_iso.unionByName(v_br_full).unionByName(v_br_short)

# 4.7) quantidade: tratar valores inválidos
v_all = v_all.withColumn(
    "quantidade",
    F.when((F.col("quantidade").isNull()) | (F.col("quantidade") == "") | (F.col("quantidade").rlike(r"^\s+$")), None)
    .otherwise(F.col("quantidade")).cast("int")
).withColumn(
    "produto",
    F.when((F.col("produto").isNull()) | (F.col("produto") == "") | (F.col("produto").rlike(r"^\s+$")), None)
    .otherwise(F.col("produto"))
)

# 4.8) valor_total: limpar e converter
valor_raw = F.col("valor_total")
valor_sem_ponto_milhar = F.when(valor_raw.contains(","), F.regexp_replace(valor_raw, "\\.", "")).otherwise(valor_raw)
valor_ponto_decimal = F.regexp_replace(valor_sem_ponto_milhar, ",", ".")
valor_sem_espacos = F.regexp_replace(valor_ponto_decimal, " ", "")
v_all = v_all.withColumn("valor_total", valor_sem_espacos.cast(T.DecimalType(10, 2)))

vendas_silver = v_all

print("--- Schema (SILVER / depois) ---")
vendas_silver.printSchema()

print("--- Amostra (SILVER / depois) ---")
# d(vendas_silver.limit(20))
d(vendas_silver)
print(vendas_silver.count())



In [0]:
# 5) Checagens rápidas (antes e depois)
"""
PASSO: Checagens mínimas para não se enganar
VALIDAR:
  - Contagem não mudou (aqui não filtramos/removemos linhas)
  - data_pedido parseou (nulos indicam formato inesperado)
  - UFs: sem valores fora da lista
  - status: só 3 categorias
"""

print("Linhas BRONZE:", vendas_bronze.count())
print("Linhas SILVER:", vendas_silver.count())

# Nulos nas colunas-chave
cols_key = ["id_pedido", "data_pedido", "uf", "produto", "quantidade", "valor_total", "status"]
null_counts = vendas_silver.select([
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in cols_key
])
d(null_counts)

# Domínio de UF
ufs_validas = [
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA",
    "PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"
]
ufs_invalidas = vendas_silver.filter(~F.col("uf").isin(ufs_validas)).select("uf").distinct()
print("UFs inválidas (deve ficar vazio):")
d(ufs_invalidas)

# Distribuição de status
print("Distribuição de status:")
d(vendas_silver.groupBy("status").count().orderBy(F.desc("count")))


In [0]:
# Quick fix
# Se houverem linhas com valores nulos, elas são removidas aqui
null_cols = []
vendas_silver_not_null = vendas_silver
colunas_que_nao_podem_ser_nulas = ['quantidade' , 'produto']
for col, val in zip(null_counts.columns, null_counts.collect()[0]):
    if val > 0:
        null_col_name = col.replace('null_', '')
        null_cols.append(null_col_name)
        print(f"Coluna com valor nulo: {null_col_name}")
        vendas_silver_not_null = vendas_silver_not_null.filter(F.col(null_col_name).isNotNull())

d(vendas_silver_not_null)

In [0]:
# 6) Antes vs Depois (comparação final rápida)
"""
PASSO: Comparar antes vs depois
O QUE OBSERVAR:
  - Nomes de colunas consistentes
  - Tipos: data_pedido TIMESTAMP, quantidade INT, valor_total DECIMAL
  - Strings mais previsíveis (uf/status padronizados)
"""

print("--- BRONZE (schema) ---")
vendas_bronze.printSchema()

print("--- SILVER (schema) ---")
vendas_silver_not_null.printSchema()

print("Amostra BRONZE:")
d(vendas_bronze)

print("Amostra SILVER:")
d(vendas_silver_not_null)


In [0]:
vendas_silver_not_null.createOrReplaceTempView("vendas_silver")

In [0]:
print(vendas_bronze.select('UF ').distinct().count())
print(vendas_silver_not_null.select('UF').distinct().count())

In [0]:
%sql
SELECT 
  TRUNC(data_pedido, 'MM') DATA_PEDIDO_MES, 
  produto, 
  SUM(VALOR_TOTAL) AS VALOR_TOTAL_VENDA, 
  COUNT(*) AS TOTAL_PRODUTOS_VENDIDOS, 
  FORMAT_NUMBER(SUM(VALOR_TOTAL)/ COUNT(*), 3) AS AVG_VALOR
FROM vendas_silver
GROUP BY ALL
ORDER BY 3 DESC

In [0]:
%sql
-- DROP SCHEMA aulas_ao_vivo.live_20260302 CASCADE;